# Explode Functions in Databricks PySpark

The `explode` family of functions transforms array and map columns into multiple rows, making nested data easier to analyze.

## Core Explode Functions

### 1. **explode()**
Expands an array or map into multiple rows. Creates one row per array element or map key-value pair.

**Syntax:**
```python
from pyspark.sql.functions import explode
df.select(explode(col("array_column")).alias("element"))
```

**Behavior:**
* **Arrays**: Creates one row per element
* **Maps**: Creates one row per key-value pair (produces two columns: `key` and `value`)
* **NULL values**: Rows with NULL arrays/maps are **excluded** from results
* **Empty arrays**: Rows with empty arrays are **excluded** from results

### 2. **explode_outer()**
Same as `explode()` but preserves rows with NULL or empty arrays/maps.

**Syntax:**
```python
from pyspark.sql.functions import explode_outer
df.select(explode_outer(col("array_column")).alias("element"))
```

**Behavior:**
* Keeps rows where array/map is NULL (exploded column becomes NULL)
* Keeps rows where array is empty (exploded column becomes NULL)
* Use this when you need to preserve all original rows

### 3. **posexplode()**
Like `explode()` but also includes the position/index of each element.

**Syntax:**
```python
from pyspark.sql.functions import posexplode
df.select(posexplode(col("array_column")).alias("pos", "element"))
```

**Output:**
* Creates two columns: `pos` (0-based index) and `col` (element value)
* For maps: `pos` is the key, `col` is the value

### 4. **posexplode_outer()**
Combines `posexplode()` with NULL/empty preservation.

## Array-Specific Functions

### **flatten()**
Flattens nested arrays (array of arrays) into a single-level array.

```python
from pyspark.sql.functions import flatten
df.select(flatten(col("nested_array")).alias("flat_array"))
```

**Example:** `[[1,2], [3,4]]` → `[1,2,3,4]`

### **arrays_zip()**
Combines multiple arrays element-wise into an array of structs.

```python
from pyspark.sql.functions import arrays_zip
df.select(arrays_zip(col("names"), col("ages")).alias("zipped"))
```

**Example:** `["Alice", "Bob"]` + `[25, 30]` → `[{names: "Alice", ages: 25}, {names: "Bob", ages: 30}]`

## Common Use Cases

### Transform JSON Arrays
```python
# Input: {"items": ["apple", "banana", "orange"]}
df.select(col("id"), explode(col("items")).alias("item"))
# Output: Multiple rows with id and individual items
```

### Unnest Nested Structures
```python
# Flatten array of structs
df.select(col("order_id"), explode(col("line_items")).alias("item")) \
  .select(col("order_id"), col("item.product"), col("item.quantity"))
```

### Preserve All Rows (Even Empty Arrays)
```python
# Use explode_outer to keep customers with no orders
df.select(col("customer_id"), explode_outer(col("orders")).alias("order"))
```

### Track Element Position
```python
# Get array index along with values
df.select(col("id"), posexplode(col("events")).alias("event_index", "event"))
```

## Important Notes

* **Performance**: `explode` can significantly increase row count; filter before exploding when possible
* **NULL handling**: Choose between `explode` (excludes NULLs) and `explode_outer` (keeps NULLs) based on your needs
* **Multiple explodes**: Chaining multiple explodes creates a **cartesian product** of elements
* **Alternative**: Consider `inline()` for exploding arrays of structs into columns instead of rows

## Comparison Table

| Function          | Keeps NULLs? | Keeps Empty? | Position Index? | Use Case                          |
| ----------------- | ------------ | ------------ | --------------- | --------------------------------- |
| `explode`         | No           | No           | No              | Standard array expansion          |
| `explode_outer`   | Yes          | Yes          | No              | Preserve all rows                 |
| `posexplode`      | No           | No           | Yes             | Need element index                |
| `posexplode_outer`| Yes          | Yes          | Yes             | Preserve rows + need index        |
| `flatten`         | N/A          | N/A          | No              | Unnest nested arrays              |
| `inline`          | N/A          | N/A          | No              | Explode structs to columns        |

### Create a dataframe with array column

In [0]:
array_appliance = [
  ('raja', ['tv', 'oven', 'ac', 'refrigerator']),
  ('raghu', ['ac', 'washing machine', None]),
  ('ram', ['tv', 'grinder']),
  ('ramesh', ['refrigerator', 'tv', 'ac', None]),
  ('ravi', None)
]

schema = 'name string, appliances array<string>'

df_app = spark.createDataFrame(array_appliance, schema)
df_app.show(truncate=False)

df_app.printSchema()

### Create a dataframe with map column

In [0]:
map_brand = [
  ('raja', {'tv': 'lg', 'oven' : 'ge', 'ac': 'haier', 'refrigerator': 'samsung'}),
  ('raghu', {'ac': 'haier', 'washing machine': 'whirlpool'}),
  ('ram', {'tv': 'samsung', 'grinder': 'bajaj'}),
  ('ramesh', {'refrigerator': 'samsung', 'tv': 'lg', 'ac': 'haier'}),
  ('ravi', None)
]

schema = 'name string, brand map<string,string>'

df_brand = spark.createDataFrame(map_brand, schema)
df_brand.show(truncate=False)

display(df_brand)

df_brand.printSchema()

### Explode array field

In [0]:
from pyspark.sql.functions import explode
from pyspark.sql.functions import count_distinct
from pyspark.sql.functions import col
from pyspark.sql.functions import size, expr
from pyspark.sql.functions import lit

In [0]:
df_app_exploded = df_app.select("name",explode("appliances").alias("appliances"))

display(df_app_exploded)                               

In [0]:
df_app_exploded.groupBy("name").agg(count_distinct("appliances").alias("appliances_count")).show()

In [0]:
df_app.select("name", size(expr("filter(appliances, x -> x IS NOT NULL)")).alias("appliances_count")).show()

In [0]:
df_brand_exploded = df_brand.select("name", explode("brand").alias("appliances", "brand"))

df_brand_exploded.show()

df_brand_exploded.printSchema()

In [0]:
df_brand_exploded.groupBy("name")\
    .agg(count_distinct(expr("CASE WHEN appliances IS NOT NULL AND appliances != '' THEN appliances END")).alias("appliances_count"),\
    count_distinct(expr("CASE WHEN brand IS NOT NULL AND brand != '' THEN brand END")).alias("brand_count")).show()

### Explode outer to consider NULL values

In [0]:
from pyspark.sql.functions import explode_outer

In [0]:
df_app_exploded_outer = df_app.select('name', explode_outer(col('appliances')).alias('appliances'))

df_app_exploded_outer.show()

df_app_exploded_outer.printSchema()

In [0]:
df_brand_exploded_outer = df_brand.select("name","brand", explode_outer(col("brand")).alias("appliances", "brand_name"))

df_brand_exploded_outer.show()

df_brand_exploded_outer.printSchema()

### Positional Explode

In [0]:
from pyspark.sql.functions import posexplode, posexplode_outer

In [0]:
df_app.select("name", posexplode(col("appliances")).alias("position", "appliances")).show()

In [0]:
df_brand.select("name", posexplode(col("brand")).alias("position", "appliances", "brand")).show()

In [0]:
df_app.select("name", posexplode_outer(col("appliances")).alias("position", "appliances")).show()

In [0]:
df_brand.select("name", posexplode_outer(col("brand")).alias("position", "appliances", "brand")).show()